In [1]:
%pip install voyageai

  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp313-cp313-win_amd64.whl.metadata (21 kB)
   ---------------------------------------- 0.0/577.1 kB ? eta -:--:--
   ---------------------------------------- 577.1/577.1 kB 4.2 MB/s  0:00:00
   ---------------------------------------- 0.0/933.7 kB ? eta -:--:--
   ---------------------------------------- 933.7/933.7 kB 12.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 26.3 MB/s eta 0:00:01
   ---- ----------------------------------- 1.3/12.6 MB 26.3 MB/s eta 0:00:01
   ------------ --------------------------- 3.9/12.6 MB 8.2 MB/s eta 0:00:02
   ------------------ --------------------- 5.8/12.6 MB 8.4 MB/s eta 0:00:01
   ------------------------- -------------- 7.9/12.6 MB 8.6 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.6 MB 8.4 MB/s eta 0:00:01
   -------------------------


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import math
import time
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

Here client is the Voyage client, so in this notebook it replaces the Anthropic client. Do not mix them.

## the embedding function

In [6]:
def embed_many(texts, model="voyage-3-large", input_type="document"):
    """Embeds a whole list of texts in ONE request."""
    result = client.embed(texts, model=model, input_type=input_type)
    return result.embeddings

def generate_embedding(text, model="voyage-3-large", input_type="query"):
    """Embeds one text (one request)."""
    return embed_many([text], model=model, input_type=input_type)[0]

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

In [7]:
chunks = [
    "Engineers fixed 247 bugs this year.",
    "Researchers studied the XDR-47 strain in three hospitals.",
    "The cafeteria menu changes every Monday.",
]
question_text = "How many bugs did the engineers fix?"

## see “meaning” in action

In [8]:
time.sleep(60)  # lets the per-minute limit reset after earlier attempts

chunk_embeddings = embed_many(chunks, input_type="document")      # request 1
question = generate_embedding(question_text, input_type="query")  # request 2

print("Embedding length:", len(question))
print("First 5 numbers:", [round(x, 4) for x in question[:5]])
print()

scores = [(cosine(question, e), c) for e, c in zip(chunk_embeddings, chunks)]
for score, c in sorted(scores, reverse=True):
    print(round(score, 3), "-", c)

Embedding length: 1024
First 5 numbers: [-0.0481, -0.0072, -0.0058, -0.0156, 0.0335]

0.719 - Engineers fixed 247 bugs this year.
0.351 - Researchers studied the XDR-47 strain in three hospitals.
0.242 - The cafeteria menu changes every Monday.
